### 線形モデルのReCo結晶のキュリー温度回帰モデル全探索


In [ ]:
!ls


**データ取得からデータ解析**


In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV
import warnings
from sklearn.model_selection import cross_val_score, KFold
import pandas as pd
import itertools
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

try:
    import progressbar
    g_have_progressbar = True
except:
    g_have_progressbar = False

pd.set_option("display.max_rows", 10)
pd.set_option("display.max_columns", 60)
warnings.filterwarnings("ignore")


In [ ]:
g_data_name = "ReCo"  # ReCo or Carbon8
g_regressionmodel = "Linear"  # Linear, Ridge, RF


Re-Co 結晶のキュリー温度TCが入ってるデータセットを選択します。

確認のため中身を表示します。

In [ ]:
def get_data(data_name="ReCo"):
    """get data from a file.

    Args:
        data_name (str, optional): data name. Defaults to "ReCo".

    Raises:
        ValueError: unknown data_name.

    Returns:
        pd.DataFrame: data.
        [str]: a list of explanatory variable names.
        str: target variable name.
    """
    if data_name == "ReCo":
        df = pd.read_csv("../data/TC_ReCo_detail_descriptor.csv")
        descriptor_names = ['C_R', 'C_T', 'vol_per_atom', 'Z', 'f4', 'd5', 'L4f', 'S4f', 'J4f',
                            '(g-1)J4f', '(2-g)J4f']
        target_name = 'Tc'
    elif data_name == "Carbon8":
        df = pd.read_csv("../data_calculated/Carbon8_cell_descriptor_Etot.csv")
        descriptor_names = ['a0.25_rp1.0', 'a0.25_rp1.5', 'a0.25_rp2.0', 'a0.25_rp2.5',
                            'a0.25_rp3.0', 'a0.5_rp1.0', 'a0.5_rp1.5', 'a0.5_rp2.0', 'a0.5_rp2.5',
                            'a0.5_rp3.0', 'a1.0_rp1.0', 'a1.0_rp1.5', 'a1.0_rp2.0', 'a1.0_rp2.5',
                            'a1.0_rp3.0', ]
        target_name = 'Etot'
    else:
        raise ValueError("unknown data_name={}".format(data_name))
    return df, descriptor_names, target_name


g_df, g_descriptor_names, g_target_name = get_data(g_data_name)


説明変数と目的変数を選択します。

In [ ]:
g_Xraw = g_df[g_descriptor_names].values
g_y = g_df[g_target_name].values


In [ ]:
from sklearn.preprocessing import StandardScaler
g_scaler = StandardScaler()
g_scaler.fit(g_Xraw)
g_X = g_scaler.transform(g_Xraw)


全探索組み合わせiteratorをつくります。

In [ ]:
def all_combinations(n, m=None):
    """make the iterator of all combinations

    Args:
        n (int): the number of descriptors
        m (int, optional): the maximum number of descriptors. Defaults to None.

    Yields:
        set: a set of descriptors
    """
    seq = range(n)
    if m is None:
        m = n
    for i in range(1, m+1):
        for x in itertools.combinations(seq, i):
            yield x


回帰能を計算する関数を定義し、クロスバリデーションを行います。


In [ ]:
from sklearn.metrics import make_scorer


def fit_cv_X(x, y, mode="Linear", nfold=5, nfold_model=3):
    """make CV scores 

    Args:
        x (np.array): descriptor
        y (np.array): target values
        mode (str, optional): a type of regression. Defaults to "Linear".
        nfold (int, optional): the number of foldings in linear regression. Defaults to 5.
        nfold_model (int, optional): the number foldings in RdigeCV. Defaults to 3.

    Raises:
        ValueError: unknown mode

    Returns:
        float: the mean value of the score
        float: the stddev vlaue of the score
        np.array: coefficients of the regression for linear models, feature importance for randomforest mdoel.
    """
    kf = KFold(n_splits=nfold, shuffle=True, random_state=6)
    kf_model = KFold(n_splits=nfold_model, shuffle=True, random_state=6)

    meanlist = []
    varlist = []

    if mode == "Linear":
        reg = LinearRegression(fit_intercept=True)
    elif mode == "Ridge":
        reg = RidgeCV(cv=kf_model, fit_intercept=True)
    elif mode == "RF":
        reg = RandomForestRegressor(n_estimators=10)
    else:
        raise ValueError("unknown mode=", mode)

    scorelist = cross_val_score(
        reg, x, y, scoring=make_scorer(r2_score), cv=kf)

    # 平均
    mean = np.mean(scorelist)
    # 標準偏差
    std = np.std(scorelist)

    # モデルを作り直す。
    reg.fit(x, y)

    if mode in ["Linear", "Ridge"]:
        return mean, std, reg.coef_
    elif mode == "RF":
        return mean, std, reg.feature_importances_
    else:
        raise ValueError("unknown mode=", mode)


In [ ]:
def fit_and_predict_combinations(x, y, regressionmodel, descriptor_names, have_progressbar=False, max_component=None):
    """accumulate the result of exhaustive search.

    Args:
        x (np.array): descriptor
        y (np.array): target value
        regressionmodel (str): regression model name
        descriptor_names ([str]): 説明変数名リスト。
        have_progressbar (bool, optional): have progress bar. Defaults to False.
        max_component (int, optional): the maximum number of descriptors. Defaults to None.

    Returns:
        dict: results.  a list of combination, mean,variance,coefficient
    """
    print_indicatorlabel = False

    n = x.shape[1]
    if max_component is None:
        max_component = n

    combi_list = []
    mean_list = []
    std_list = []
    coef_list = []

    for ncombi, s in enumerate(all_combinations(n, max_component)):
        pass

    if have_progressbar:
        bar = progressbar.ProgressBar(max_value=ncombi+1)

    for i, icombi in enumerate(all_combinations(n, max_component)):
        if have_progressbar:
            bar.update(i+1)

        icombi = np.array(icombi)
        combi = np.array(descriptor_names)[np.array(icombi)]
        combi_list.append(icombi)
        if print_indicatorlabel:
            print("indicators", combi)
        xtry = x[:, icombi]
        ytry = y
        mean, std, coef = fit_cv_X(xtry, ytry, regressionmodel)
        mean_list.append(mean)
        std_list.append(std)
        # The first element　of coef is the coefficient to y
        coef_list.append(coef.ravel())

    mean_list = np.array(mean_list)
    std_list = np.array(std_list)

    return {"combination": combi_list, "score_mean": mean_list, "score_std": std_list, "coef": coef_list}


実行する。
少々時間がかかります。

結果をmean valueでsortします。

In [ ]:
import pickle
import os

def save_df(df_result, descriptor_names, regressionmodel, savefile):
    """save dataframe as pickle.

    Args:
        df_result (pd.DataFrame): result data.
        descriptor_names ([str]]): a list of explanatory variable names.
        regressionmodel (LinearModel|ForestRegressor): regression model.
        savefile (str): a filename of pickle.
    """
    # print(descriptor)
    descriptor_names = np.array(descriptor_names)
    # df_result.to_csv("TC_ReCo_ES.csv")
    combinationlist = []
    for x in df_result["combination"].values:
        x2 = descriptor_names[np.array(x)]
        combinationlist.append("|".join(x2))
    df_result["descriptor"] = combinationlist
    with open(savefile, "wb") as f:
        pickle.dump(df_result, f)


os.makedirs("models", exist_ok=True)
g_savefile = "models/ESresult_{}_{}_{}.pickle".format(
    g_regressionmodel, g_target_name, g_data_name)
print("filename", g_savefile)
if not os.path.exists(g_savefile):
    g_result = fit_and_predict_combinations(g_X, g_y, g_regressionmodel,
                                            g_descriptor_names,
                                            have_progressbar=g_have_progressbar)
    g_df_score = pd.DataFrame(g_result).sort_values(
        by="score_mean", ascending=False).reset_index(drop=True)

    save_df(g_df_score, g_descriptor_names, g_regressionmodel, g_savefile)

with open(g_savefile, "rb") as f:
    g_df_result = pickle.load(f)
    print("load", g_savefile)
g_df_score = g_df_result[['combination', 'score_mean', 'score_std', 'coef']]


**可視化**

DOSの表示を行います。

In [ ]:
def show_r2_hist(df, xlim=None,filename=None):
    """show R2 as a histgram.

    Args:
        df (pd.DataFrame): data.
        xlim ((float,float), optional): x lim of plot. Defaults to None.
        filename (str, optional): a filename. Defaults to None.
    """
    fig, ax = plt.subplots()
    df.hist("score_mean", bins=100, ax=ax)
    ax.set_xlabel("$R^2$")
    ax.set_ylabel("DOS")
    if xlim is not None:
        ax.set_xlim(xlim)
    fig.tight_layout()
    if filename is not None:
        fig.savefig(filename)
        print("saved to",filename)


os.makedirs("image_executed", exist_ok=True)
show_r2_hist(g_df_score,filename="image_executed/ReCo_ES_score_vs_DOS.png")


In [ ]:
# 拡大
show_r2_hist(g_df_score, xlim=(0.8, 1.0))


In [ ]:
def calculate_coeffix(descriptor, combilist, coeflist):
    """表示のために 係数０の部分を加えて係数を作りなおす。

    Args:
        descriptor (list): all the descriptor names
        combilist (list): a list of descriptor combinations of the models
        coeflist (np.array): a list of coefficients of the models

    Returns:
        list: a list of coefficnets whose length is the same as the length of all the descriptors
    """
    n = len(descriptor)
    coeffixlist = []
    for combi, coef in zip(combilist, coeflist):

        coeffix = np.zeros((n))
        # if combi=[1,2], and coef=[val1,val2], then coeffix=[0,val1,val2,0,0]
        for i, id in enumerate(combi):
            coeffix[id] = coef[i]

        # 都合でlistに直す。
        coeffixlist.append(list(coeffix))
    return coeffixlist


g_coeffixlist = calculate_coeffix(g_descriptor_names,
                                  g_df_score["combination"].values, g_df_score["coef"].values)
g_df_coef = pd.DataFrame(g_coeffixlist, columns=g_descriptor_names)
g_df_result = pd.concat([g_df_score, g_df_coef], axis=1)


In [ ]:
g_df_result

In [ ]:
import seaborn as sns


def show_weight_diagram(df_result, descriptor_names, nmax=50):
    """weight diagramの表示

    Args:
        df_result (pd.DataFrame): data
        nmax (int, optional): the maximum number of the data to show. Defaults to 50.
    """
    x = df_result.loc[:nmax, descriptor_names].values
    x = np.log10(np.abs(x))
    df_x = pd.DataFrame(x, columns=descriptor_names).replace(
        [-np.inf, np.inf], np.nan)
    df_weight_diagram = df_x.fillna(-3)
    fig, ax = plt.subplots()
    ax.set_title("log10(abs(coef))")
    sns.heatmap(df_weight_diagram.T, ax=ax)
    ax.set_ylim((-0.5, df_weight_diagram.shape[1]+0.5))
    fig.tight_layout()
    fig.savefig("image_executed/ReCo_ES_index_vs_abscoef.png")


show_weight_diagram(g_df_result, g_descriptor_names)


#### ReCoの場合
上を見ると、上位５０位の重要性は上からC_R, C_T, Sf4になるように見えます。

RFでは線形モデルの場合と異なり、C_Tのfeature_importanceが低いことが分かります。

In [ ]:
def show_indicator_diagram(df_result, descriptor_names, nmax=50):
    """indicator diagramの表示

    Args:
        df_result (pd.DataFrame): data
        nmax (int, optional): the maximum number of the data to show. Defaults to 50.
    """
    x = df_result[descriptor_names].values != 0
    df_indicator_diagram = pd.DataFrame(x, columns=descriptor_names)
    fig, ax = plt.subplots()
    sns.heatmap(df_indicator_diagram.loc[:nmax, :].T, ax=ax)
    ax.set_ylim((-0.5, df_indicator_diagram.shape[1]+0.5))
    fig.tight_layout()
    return df_indicator_diagram


g_df_indicator_diagram = show_indicator_diagram(
    g_df_result, g_descriptor_names)


In [ ]:
g_df_result.loc[:50, :].plot(y="score_mean", yerr="score_std")


ほぼ同じ予測性能を示す近似解が多数あることが分かり、R2の標準偏差を見ると分かる通り実質これらは全部同じ解です。

#### ReCoの場合
例えば、最良モデルはvol_per_atomを含みません。しかし次善モデルはvol_per_atomを含みます。乱数によっては順位が逆転する可能性もあるでしょう。
最も予測性能値が高いモデルだけで考えると、「重要な」説明変数を見逃しかねません。

occurenceは、例えば、全部取ったら全て同じ値になるので恣意的結果にならないように扱いに注意が必要です。

#### DOSのpeakで区切った解析

DOSに幾つかのpeakが見えました。それらの特徴を解析します。

In [ ]:
def make_counts(df_result, descriptor_names, sentense, ratio=False):
    """
    説明変数が用いられた回数を計算する。

    Args:
        df_result (pd.DataFrame): データ
        descriptor_names ([str]): 説明変数名リスト。
        sentense (str): query文
        ratio (bool, optional): 回数(False), 割合(True)を返す。 Defaults to False.

    Returns:
        pd.DataFrame: 回数もしくは割合データ。
    """
    x = df_result[descriptor_names].values != 0  # 係数が０でない。＝その説明変数が含まれるモデル。
    df_indicator_diagram = df_result.copy()
    df_indicator_diagram.loc[:, descriptor_names] = x

    dfq = df_indicator_diagram.query(sentense)
    print("all=", dfq.shape[0])
    if ratio:
        return np.sum(dfq[descriptor_names], axis=0)/dfq.shape[0]
    else:
        return np.sum(dfq[descriptor_names], axis=0)


def make_block_weight_list(df_result, descriptor_names, querylist):
    """
    querylistのblock weight diagramを計算する。

    Args:
        df_result (pd.DataFrame): データ。
        descriptor_names ([str]): 説明名リスト。
        querylist ([str]): query文リスト。

    Returns:
        pd.DataFrame: block weight diagram.
    """
    result = []
    for sentense in querylist:
        # 前の図に合わせるためにdescriptor_namesの順序を逆にする。
        t = make_counts(
            df_result, descriptor_names[::-1], sentense, ratio=True)
        result.append(t)
    dfq = pd.DataFrame(result, index=querylist)
    display(dfq)
    sns.heatmap(dfq.T)  # 前の図に合わせるためにtransposeする。


if g_data_name == "ReCo":
    if g_regressionmodel == "Linear":
        g_querylist = ["score_mean<0.15", "score_mean>0.15 and score_mean<0.5",
                       "score_mean>0.5 and score_mean<0.7", "score_mean>0.7"]
        make_block_weight_list(g_df_result, g_descriptor_names, g_querylist)
    if g_regressionmodel == "RF":
        g_querylist = ["score_mean<0.0",
                       "0.6<score_mean<0.8", "score_mean>0.8", ]
        make_block_weight_list(g_df_result, g_descriptor_names, g_querylist)


頻度（確率ともみなせる）~0.5の変数はその領域のモデル集合どこかでまんべんなく用いられていると思って良いでしょう。確率0.5と異なる変数から以下が言えます。

#### 線形モデルの場合
- 0.7 < score_mean : ほぼC_Rを使う。しかし、必ず使用するわけではない。
- 0.4 < score_mean< 0.7 : C_Rとvol_per_atomを使わない。C_Tを必ず使う。
- 0.15 < score_mean< 0.4 : C_RとC_Tを使わない。vol_per_atomを必ず使う。
- socre_mean < 0.15 : C_R, C_T, vol_per_atomをほぼ使わない

ということが分かる。この解析によると
重要性の順位は単一で考えるものではなく幾つか組み合わせて考えるほうが自然に思える。
説明変数C_Rは他の説明変数との間の相関があるため他の説明変数の線形結合によりある程度の変えが効くのだろう。
説明変数C_Rを使用しないと回帰能が下がる。


#### RFの場合
1: "score_mean>0.8", 2: "0.6<score_mean<0.8", 3: "score_mean<0.0"と変えていくと
1. C_RとC_Tを用いるモデル
2. C_RとC_Tを用いず、vol_per_atomを用いるモデル
3. C_RとC_Tとvol_per_atomを用いないモデル

と変わっていきます。


ランダムフォレスト回帰モデル集合の重要性を上位の頻度から解析すると、上から、C_R, C_T, vol_per_atom、その他の順になります。


#### 上位の頻度を用いた解析

DOSで区切るのではなく、regionsizeごとにモデルを区切ってみます。

In [ ]:
def make_df_by_index(df_indicator_diagram, descriptor_names, index):
    """make dataframe by index.

    Args:
        df_indicator_diagram (pd.DataFrame): data.
        descriptor_names ([str]]): a list of explanatory variable names.
        index ([str]]): a list of columns.

    Returns:
        pd.DataFrame: data.
    """
    dfq = df_indicator_diagram.iloc[index, :]
    print("all=", dfq[descriptor_names].shape[0])
    df_all = pd.DataFrame({"N": [dfq[descriptor_names].shape[0]]},)
    dfq_sum = dfq[descriptor_names].astype(int).sum(axis=0)
    df1 = pd.DataFrame(dfq_sum).T

    return pd.concat([df1, df_all], axis=1)
    # print(np.sum(dfq[descriptor_names], axis=0))


def make_all_ind_by_index(df_indicator_diagram: pd.DataFrame, descriptor_names: [str], regionindex: [int], regionsize: int):
    """make dataframe by regionindex.

    Args:
        df_indicator_diagram (pd.DataFrame): data.
        descriptor_names ([str]]): a list of explanatory variables.
        regionindex ([int]): a list of region index.
        regionsize (int): region size.

    Returns:
        pd.DataFrame: data.
    """
    df_ind_list = []
    for i in regionindex:
        region = list(range(i*regionsize, (i+1)*regionsize))
        df_ind = make_df_by_index(
            df_indicator_diagram, descriptor_names, region)
        df_ind_list.append(df_ind)
    _df = pd.concat(df_ind_list, axis=0).reset_index(drop=True)

    names = list(_df.columns)
    names.remove("N")
    v0 = _df["N"]
    for name in names:
        _df[name] = _df[name]/v0
        
    if False:
        fig, ax = plt.subplots()
        _df[names].T.plot(ax=ax)
        ax.set_ylabel("frequency")
        ax.set_xticks(list(range(len(names))))
        ax.set_xticklabels(names, rotation=90)
        ax.set_ylim((0, 1))
    return _df


g_regions = [_i for _i in range(5)]
g_regionsize = 300
g_df_imp_by_index = make_all_ind_by_index(
    g_df_indicator_diagram, g_descriptor_names, g_regions, g_regionsize)


In [ ]:
def show_r2_by_index(df_result, regions, regionsize):
    """
    regionsize*i番目のR2を表示する。
    
    Args:
        df_result (pd.DataFrame): data.
        regions ([int]): a list of regions.
        regionsize (int): region size.
    """
    fig, ax = plt.subplots()
    regions = np.array(regions)
    for i in regions:
        dfp = df_result.loc[[regionsize*i], :]
        dfp.plot(y="score_mean", yerr="score_std", ax=ax,label="yerr at {}th".format(regionsize*i))
    dfp1 = df_result.loc[regionsize*regions, ["score_mean"]]
    dfp1.plot(y="score_mean", ax=ax)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.set_ylabel("R2")
    ax.set_xlabel("Nst model")


show_r2_by_index(g_df_result, g_regions, g_regionsize)


In [ ]:
from matplotlib.ticker import MaxNLocator


def show_df_imp_by_index(df_imp_by_index, descriptor_names, regions, regionsize):
    """
    各領域の説明変数の頻度を表示する。
    
    Args:
        df_result (pd.DataFrame): data.
        regions ([int]): a list of regions.
        regionsize (int): region size.    
    """
    xticks_str = []
    for i in regions:
        xticks_str.append("[{}:{}]".format(i*regionsize, (i+1)*regionsize))
    fig, ax = plt.subplots()
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    df_imp_by_index[descriptor_names].plot(marker="o", ax=ax)
    ax.set_xticks(list(range(len(regions))))
    ax.set_xticklabels(xticks_str)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.set_ylabel("frequency")


show_df_imp_by_index(g_df_imp_by_index, g_descriptor_names,
                     g_regions, g_regionsize)



##### RFの場合
上の解析だと[0:600]程度はC_Rが重要、S4f, f4, Zが次に重要。C_T, vol_per_atomは重要ではない。
そして、[900:1500]ではC_Tが重要である、C_R,S4fは重要では無い、となります。


回帰モデルの重要性はモデルによります。
そしてモデルは全て近似解でほぼ同じ回帰性能を示すモデルが多数ありますので、「重要性」を議論する際は定義してから議論してください。

### 問題１

（model全探索でない）LASSOの、係数と回帰性能の比較

### 問題２

（itemset miningを行った後に）itemset miningによる共通説明変数の抽出。